In [3]:
# 示例用法
if __name__ == "__main__":
    import sys
    
    
    log_file_path = "/disk6T/ypguo/async-FL/时间仿真/output (6).log"
    delay_file_path = "/disk6T/ypguo/async-FL/时间仿真/42.7_DelayAssociationGroup.csv"
    
    # 如果提供了常数t，则使用它
    constant_t = 0  # 默认值
    # if len(sys.argv) > 3:
    #     try:
    #         constant_t = float(sys.argv[3])
    #     except ValueError:
    #         print(f"警告: 无法将 '{sys.argv[3]}' 转换为浮点数，使用默认常数t=5.0")
    
    # 分析日志并保存结果
    results = analyze_fed_log_with_simulation_time(log_file_path, delay_file_path, constant_t=constant_t)
    
    # 打印部分结果作为示例
    print("\n分析结果示例 (前5行):")
    print(results.head())

分析结果已保存到: /disk6T/ypguo/async-FL/时间仿真/output (6)_with_simulation_time.csv

分析结果示例 (前5行):
   Epoch  Group_ID                         Selected_Clients   Accuracy  \
0      1         4   [2, 25, 7, 35, 75, 86, 92, 80, 67, 88]  10.086138   
1      2         0   [11, 1, 6, 77, 83, 68, 34, 69, 52, 32]  11.969151   
2      3         1   [56, 66, 4, 40, 99, 49, 15, 14, 71, 5]  12.820513   
3      4         2  [36, 18, 0, 23, 24, 78, 60, 82, 41, 64]  17.748397   
4      5         3   [48, 45, 3, 28, 30, 47, 65, 20, 8, 17]  14.923878   

       Loss   Run_Time  Simulation_Time  edge_round_time  
0  3.596663  24.921752        15.449481        15.449481  
1  2.843316  30.030816        15.042202        15.042202  
2  3.119751  35.357575        23.006787        23.006787  
3  2.813778  41.265135        16.731804        16.731804  
4  2.491911  47.874153        17.063551        17.063551  


In [4]:
import re
import pandas as pd
import os
import numpy as np
from collections import defaultdict

def analyze_dynamic_delay_log(log_file_path, output_path=None, constant_t=5.0):
    """
    Parses a federated learning log file with dynamic client delays,
    calculates simulation time for each epoch, and saves the results to a CSV file.
    (Includes debugging print statements)

    Args:
        log_file_path (str): The path to the log file.
        output_path (str, optional): The path to save the output CSV file.
                                     If None, generates a default name based on the log file name.
        constant_t (float, optional): Constant time added in simulation time calculation.
                                      Defaults to 5.0.

    Returns:
        pandas.DataFrame: DataFrame containing the parsed and calculated results.
                          Returns None if the file cannot be read or parsing fails significantly.
    """
    print(f"--- Starting analysis for: {log_file_path} ---") # DEBUG Start
    try:
        # Try common encodings
        encodings_to_try = ['utf-8', 'latin-1', 'iso-8859-1']
        log_content = None
        for enc in encodings_to_try:
            try:
                with open(log_file_path, 'r', encoding=enc) as file:
                    log_content = file.read()
                print(f"Successfully read file with encoding: {enc}") # DEBUG Encoding
                break # Stop if successful
            except UnicodeDecodeError:
                print(f"Failed to decode with {enc}...") # DEBUG Encoding Fail
                continue
            except Exception as e:
                 print(f"Error reading log file with encoding {enc}: {e}")
                 # Don't return yet, try next encoding

        if log_content is None:
             print(f"Error: Could not read log file with any attempted encoding: {log_file_path}")
             return None

    except FileNotFoundError:
        print(f"Error: Log file not found at {log_file_path}")
        return None
    except Exception as e:
        print(f"Error during file access: {e}")
        return None

    # --- Stage 1: Extract all relevant pieces of information with positions ---
    print("--- Stage 1: Extracting data with regex ---") # DEBUG Stage

    # Regex patterns (Consider potential variations)
    # Make hyphen matching more flexible, allow optional space around brackets
    epoch_pattern = re.compile(
        r'group_ready_num:\s*(\d+)\s*\[.*?\]\s*\n-{2,}\s*\nEpoch\s*(\d+)\s*tested,\s*accuracy:\s*([\d\.]+)\s*loss\s*([\d\.]+)\s*run_time\s*([\d\.]+)'
    )
    selection_pattern = re.compile(r'group\s*(\d+)\s*selected_clients:\s*\[([\d,\s]*)\]') # Added \s* for flexibility
    upload_pattern = re.compile(r'Client\s*(\d+)\s*uploaded.*?simulated_delay:\s*([\d\.]+)') # Added \s*

    epochs_found = []
    selections_found = []
    uploads_found = []

    # Find all epoch completion info
    for match in epoch_pattern.finditer(log_content):
        try:
            # group_id capture here is less critical as we determine it later
            epochs_found.append({
                'match_obj': match,
                'epoch_num': int(match.group(2)),
                'accuracy': float(match.group(3)),
                'loss': float(match.group(4)),
                'run_time': float(match.group(5)),
                'end_pos': match.end() # Position where the match ends
            })
        except (ValueError, IndexError) as e:
             print(f"Warning: Could not parse epoch block at position {match.start()}: {e}")

    # Find all client selections
    for match in selection_pattern.finditer(log_content):
        try:
            group_id = int(match.group(1))
            clients_str = match.group(2).strip()
            selected_clients = [int(c.strip()) for c in clients_str.split(',') if c.strip()] if clients_str else []
            selections_found.append({
                'match_obj': match,
                'group_id': group_id,
                'selected_clients': selected_clients,
                'start_pos': match.start(), # Position where the match starts
                'end_pos': match.end()
            })
        except (ValueError, IndexError) as e:
            print(f"Warning: Could not parse selection block at position {match.start()}: {e}")

    # Find all client uploads
    for match in upload_pattern.finditer(log_content):
        try:
            uploads_found.append({
                'match_obj': match,
                'client_id': int(match.group(1)),
                'delay': float(match.group(2)),
                'start_pos': match.start() # Position where the match starts
            })
        except (ValueError, IndexError) as e:
            print(f"Warning: Could not parse upload block at position {match.start()}: {e}")

    # DEBUG: Print counts
    print(f"\nRegex Results:")
    print(f"  Found {len(epochs_found)} epoch blocks.")
    print(f"  Found {len(selections_found)} selection blocks.")
    print(f"  Found {len(uploads_found)} upload blocks.")

    # Add a check before the main processing loop
    if not epochs_found:
        print("\nError: No epoch blocks were found. Cannot proceed. Check 'epoch_pattern' regex and log format.")
        return None
    if not selections_found:
        print("\nError: No selection blocks were found. Cannot proceed. Check 'selection_pattern' regex and log format.")
        return None
    # Uploads might legitimately be zero in some logs, but issue a warning if needed
    if not uploads_found:
         print("\nWarning: No upload blocks with 'simulated_delay' were found. edge_round_time will be 0.")


    # Sort selections and uploads by their position in the log
    selections_found.sort(key=lambda x: x['start_pos'])
    uploads_found.sort(key=lambda x: x['start_pos'])

    # --- Stage 2: Correlate and Calculate ---
    print("\n--- Stage 2: Correlating data and calculating times ---") # DEBUG Stage

    results = []
    last_sim_time = defaultdict(float)
    group_epoch_counters = defaultdict(int)

    group_selections_ordered = defaultdict(list)
    for sel in selections_found:
        group_selections_ordered[sel['group_id']].append(sel)

    # Iterate through epochs chronologically based on their end position
    epochs_found.sort(key=lambda x: x['end_pos'])

    for epoch_info in epochs_found:
        print(f"\nProcessing Epoch {epoch_info['epoch_num']} (ends at {epoch_info['end_pos']})") # Debug Start Epoch Loop

        # --- Determine the correct Group ID and Selected Clients for this Epoch ---
        potential_selections = [sel for sel in selections_found if sel['end_pos'] < epoch_info['end_pos']]
        print(f"  Found {len(potential_selections)} potential selections ending before epoch end pos {epoch_info['end_pos']}.") # Debug Selections

        if not potential_selections:
            print(f"  Warning: No selection found ending before Epoch {epoch_info['epoch_num']} (ends at {epoch_info['end_pos']}). Skipping.")
            continue # Skip this epoch if no preceding selection found

        current_selection = max(potential_selections, key=lambda sel: sel['end_pos'])
        group_id = current_selection['group_id']
        print(f"  Most recent preceding selection ends at {current_selection['end_pos']}, Group ID: {group_id}") # Debug Group Determination

        # --- Use ordered selection logic based on group counter ---
        group_epoch_counters[group_id] += 1
        current_group_epoch_index = group_epoch_counters[group_id]
        print(f"  This is the {current_group_epoch_index}-th epoch recorded for group {group_id}.") # Debug Counter

        if current_group_epoch_index > len(group_selections_ordered[group_id]):
            print(f"  Warning: Mismatch - Index {current_group_epoch_index} > available selections ({len(group_selections_ordered[group_id])}) for group {group_id}. Using last available selection.")
            if not group_selections_ordered[group_id]:
                print(f"  Error: No selections recorded for group {group_id}. Cannot process Epoch {epoch_info['epoch_num']}.")
                continue # Skip if no selections ever found for this group
            selection_details = group_selections_ordered[group_id][-1]
        else:
            # Use the k-th selection record for this group (k = current_group_epoch_index)
            selection_details = group_selections_ordered[group_id][current_group_epoch_index - 1]

        selected_clients = selection_details['selected_clients']
        selection_start_pos = selection_details['start_pos']
        print(f"  Using selection starting at {selection_start_pos}: Clients {selected_clients}") # Debug Clients

        # --- Find the relevant upload delays ---
        next_selection_pos = float('inf')
        if current_group_epoch_index < len(group_selections_ordered[group_id]):
            next_selection_pos = group_selections_ordered[group_id][current_group_epoch_index]['start_pos']
        print(f"  Upload search window: [{selection_start_pos}, {next_selection_pos})") # Debug Window

        relevant_delays = []
        processed_clients_in_epoch = set()
        upload_search_count = 0
        found_upload_details = [] # Store details for debugging

        for upload in uploads_found:
            # Check if upload is within the time window
            if upload['start_pos'] >= selection_start_pos and upload['start_pos'] < next_selection_pos:
                upload_search_count +=1
                # Check if the upload is from one of the selected clients
                if upload['client_id'] in selected_clients:
                     # Check if we haven't already recorded an upload for this client in this window
                    if upload['client_id'] not in processed_clients_in_epoch:
                        relevant_delays.append(upload['delay'])
                        processed_clients_in_epoch.add(upload['client_id'])
                        found_upload_details.append(f"Client {upload['client_id']} (Pos {upload['start_pos']}): {upload['delay']}") # Debug Found Upload Detail

            # Optimization: If we've passed the window, stop searching
            if upload['start_pos'] >= next_selection_pos:
                break # Break inner loop once past the search window

        print(f"  Checked {upload_search_count} uploads in window.") # Debug Upload Summary
        if found_upload_details:
             print(f"  Found relevant uploads: {'; '.join(found_upload_details)}")
        else:
             print(f"  No relevant uploads found for selected clients in this window.")

        # Check if we found delays for all selected clients (optional, for debugging/validation)
        if len(processed_clients_in_epoch) != len(selected_clients) and selected_clients:
             print(f"  Warning: Selected {len(selected_clients)} clients {selected_clients}, but only found uploads for {len(processed_clients_in_epoch)} clients in the expected window.")


        # Calculate edge_round_time (max delay for this epoch)
        edge_round_time = 0.0
        if relevant_delays:
            edge_round_time = max(relevant_delays)
            print(f"  Calculated edge_round_time (max delay): {edge_round_time}") # Debug edge time
        elif selected_clients: # If clients were selected but no delays found
             print(f"  Warning: No matching uploads found for selected clients {selected_clients}. edge_round_time set to 0.")


        # Calculate Simulation Time
        b = last_sim_time[group_id]
        simulation_time = b + edge_round_time + constant_t
        print(f"  Calculated Simulation_Time: {b} + {edge_round_time} + {constant_t} = {simulation_time}") # Debug sim time

        # Store results
        print(f"  ==> Appending result for Epoch {epoch_info['epoch_num']}") # Debug Append
        results.append({
            'Epoch': epoch_info['epoch_num'],
            'Group_ID': group_id,
            'Selected_Clients': selected_clients,
            'Accuracy': epoch_info['accuracy'],
            'Loss': epoch_info['loss'],
            'Run_Time': epoch_info['run_time'],
            'edge_round_time': edge_round_time,
            'Simulation_Time': simulation_time
        })

        # Update the last simulation time for this group
        last_sim_time[group_id] = simulation_time

    # --- Stage 3: Create DataFrame and Save ---
    print("\n--- Stage 3: Creating DataFrame and saving ---") # DEBUG Stage

    if not results:
        # This is the error message the user saw. Print more context.
        print("Error: The 'results' list is empty after processing all found epochs.")
        print("This usually means that either no epochs were found initially, or the correlation logic failed for every epoch found (e.g., couldn't find preceding selections or relevant uploads).")
        print("Please review the debug output above, especially the regex match counts and messages within the epoch processing loop.")
        return None

    df = pd.DataFrame(results)

    # Reorder columns
    df = df[['Epoch', 'Group_ID', 'Selected_Clients', 'edge_round_time', 'Simulation_Time', 'Accuracy', 'Loss', 'Run_Time']]

    # Handle output path
    if output_path is None:
        dir_name = os.path.dirname(log_file_path)
        base_name = os.path.basename(log_file_path)
        base_name_no_ext = os.path.splitext(base_name)[0]
        output_path = os.path.join(dir_name, f"{base_name_no_ext}_dynamic_analysis.csv")

    # Save results
    try:
        # Convert list to string for CSV? safer for general compatibility
        df_save = df.copy()
        df_save['Selected_Clients'] = df_save['Selected_Clients'].astype(str)
        df_save.to_csv(output_path, index=False)
        print(f"\nAnalysis results saved to: {output_path}")
    except Exception as e:
        print(f"Error saving results to CSV: {e}")

    return df

# --- Example Usage ---
# Make sure to replace 'path/to/your/output.txt' with the actual path
# log_file = 'path/to/your/output.txt' 
# analysis_df = analyze_dynamic_delay_log(log_file, constant_t=5.0)

# if analysis_df is not None:
#     print("\n--- Final DataFrame Head ---")
#     print(analysis_df.head())
# --- Example Usage ---
# Assuming your log file is named 'federated_learning.log' in the same directory


In [7]:
import re
import pandas as pd
import os
from collections import defaultdict

def parse_dynamic_fed_log(log_file_path, output_csv_path=None):
    """
    解析具有动态延迟的联邦学习日志文件，提取每个epoch的信息并计算时间开销
    
    参数:
        log_file_path (str): 日志文件的路径
        output_csv_path (str, 可选): 输出CSV文件的路径
        
    返回:
        pandas.DataFrame: 包含解析结果的DataFrame
    """
    # 读取日志文件内容
    with open(log_file_path, 'r', encoding='utf-8') as file:
        log_content = file.read()
    
    results = []
    
    # 使用正则表达式匹配所有的group选择信息
    group_selections = {}
    for group_id in range(5):  # 假设有5个group (0-4)
        pattern = rf'group {group_id} selected_clients: \[([\d, ]+)\]'
        selections = re.findall(pattern, log_content)
        # 将每个选择转换为客户端id列表
        selections = [[int(client_id) for client_id in selection.split(', ')] for selection in selections]
        group_selections[group_id] = selections
    
    # 提取每个epoch的聚合信息
    epoch_pattern = r'group_ready_num: (\d+) .+?\n-+\nEpoch (\d+) tested, accuracy: ([\d\.]+) loss ([\d\.]+) run_time ([\d\.]+)'
    epoch_matches = re.findall(epoch_pattern, log_content, re.DOTALL)
    
    # 构建结果列表
    for match in epoch_matches:
        group_id = int(match[0])
        epoch = int(match[1])
        accuracy = float(match[2])
        loss = float(match[3])
        run_time = float(match[4])
        
        # 找到该epoch对应的客户端选择
        # 统计该group已经被选择的次数
        group_epoch_count = sum(1 for m in epoch_matches[:epoch_matches.index(match)] if int(m[0]) == group_id)
        
        # 确保我们有足够的选择记录
        if group_epoch_count < len(group_selections[group_id]):
            selected_clients = group_selections[group_id][group_epoch_count]
        else:
            selected_clients = []  # 如果没有找到对应的选择记录
        
        # 查找这些客户端的最大延迟
        max_delay = 0.0
        start_pos = log_content.find(f"Epoch {epoch} tested")
        
        # 往前查找直到找到group选择
        search_start = max(0, start_pos - 5000)  # 往前查找5000个字符
        selection_text = f"group {group_id} selected_clients: [{', '.join(map(str, selected_clients))}]"
        selection_pos = log_content.rfind(selection_text, search_start, start_pos)
        
        if selection_pos != -1:
            # 从group选择位置开始查找客户端上传信息
            search_end = start_pos
            search_text = log_content[selection_pos:search_end]
            
            # 查找所有客户端上传信息
            upload_pattern = r'Client (\d+) uploaded at time: .*; simulated_delay: ([\d\.]+)'
            upload_matches = re.findall(upload_pattern, search_text)
            
            for client_str, delay_str in upload_matches:
                client_id = int(client_str)
                delay = float(delay_str)
                if client_id in selected_clients:
                    max_delay = max(max_delay, delay)
        
        results.append({
            'Epoch': epoch,
            'Group_ID': group_id,
            'Selected_Clients': selected_clients,
            'Max_Delay': max_delay,
            'Accuracy': accuracy,
            'Loss': loss,
            'Run_Time': run_time
        })
    
    # 创建DataFrame
    df = pd.DataFrame(results)
    
    # 如果提供了输出路径，保存到CSV
    if output_csv_path and not df.empty:
        # 将客户端列表转换为字符串格式以便保存
        df_to_save = df.copy()
        df_to_save['Selected_Clients'] = df_to_save['Selected_Clients'].apply(lambda x: ';'.join(map(str, x)))
        df_to_save.to_csv(output_csv_path, index=False)
        print(f"结果已保存到: {output_csv_path}")
    
    return df

def analyze_dynamic_delays(df):
    """
    分析动态延迟数据
    
    参数:
        df (pandas.DataFrame): 包含解析结果的DataFrame
        
    返回:
        dict: 分析统计结果
    """
    if df.empty:
        return {
            'total_epochs': 0,
            'avg_max_delay': 0,
            'min_delay': 0,
            'max_delay': 0,
            'avg_clients_per_group': 0,
            'avg_accuracy': 0,
            'final_accuracy': 0,
            'group_stats': {}
        }
    
    stats = {
        'total_epochs': len(df),
        'avg_max_delay': df['Max_Delay'].mean(),
        'min_delay': df['Max_Delay'].min(),
        'max_delay': df['Max_Delay'].max(),
        'avg_clients_per_group': df['Selected_Clients'].apply(len).mean(),
        'avg_accuracy': df['Accuracy'].mean(),
        'final_accuracy': df.iloc[-1]['Accuracy'] if len(df) > 0 else 0,
        'group_stats': {}
    }
    
    # 按组统计
    for group_id in df['Group_ID'].unique():
        group_data = df[df['Group_ID'] == group_id]
        stats['group_stats'][group_id] = {
            'count': len(group_data),
            'avg_delay': group_data['Max_Delay'].mean(),
            'avg_accuracy': group_data['Accuracy'].mean()
        }
    
    return stats

# 使用示例
if __name__ == "__main__":
    # 设置文件路径
    log_file_path = "output.log"
    output_csv_path = "federated_learning_results_dynamic.csv"
    
    # 解析日志文件
    df_results = parse_dynamic_fed_log(log_file_path, output_csv_path)
    
    # 显示前几行结果
    print("\n解析结果预览:")
    print(df_results.head())
    
    # 分析统计
    stats = analyze_dynamic_delays(df_results)
    
    print("\n统计分析:")
    print(f"总共处理的epoch数: {stats['total_epochs']}")
    print(f"平均最大延迟: {stats['avg_max_delay']:.2f}")
    print(f"延迟范围: {stats['min_delay']:.2f} - {stats['max_delay']:.2f}")
    print(f"平均每组客户端数: {stats['avg_clients_per_group']:.2f}")
    print(f"平均准确率: {stats['avg_accuracy']:.2f}%")
    print(f"最终准确率: {stats['final_accuracy']:.2f}%")
    
    print("\n各组统计:")
    for group_id, group_stat in stats['group_stats'].items():
        print(f"Group {group_id}: {group_stat['count']} epochs, "
              f"平均延迟: {group_stat['avg_delay']:.2f}, "
              f"平均准确率: {group_stat['avg_accuracy']:.2f}%")

结果已保存到: federated_learning_results_dynamic.csv

解析结果预览:
   Epoch  Group_ID                          Selected_Clients  Max_Delay  \
0      1         0    [11, 1, 42, 34, 98, 10, 55, 6, 52, 77]        0.0   
1      2         1    [13, 57, 5, 4, 79, 14, 90, 37, 40, 71]        0.0   
2      3         2  [82, 24, 89, 97, 61, 36, 63, 39, 60, 84]        0.0   
3      4         3     [3, 27, 65, 47, 38, 9, 17, 54, 93, 8]        0.0   
4      5         4   [12, 75, 25, 73, 76, 43, 67, 35, 2, 53]        0.0   

    Accuracy      Loss   Run_Time  
0  19.981971  2.270937  15.729416  
1  11.678686  2.298183  16.126892  
2   9.985978  2.293131  16.568580  
3  16.526442  2.309987  16.918038  
4  10.006010  2.299994  17.323868  

统计分析:
总共处理的epoch数: 2007
平均最大延迟: 0.02
延迟范围: 0.00 - 6.37
平均每组客户端数: 10.00
平均准确率: 71.48%
最终准确率: 77.55%

各组统计:
Group 0: 423 epochs, 平均延迟: 0.03, 平均准确率: 71.05%
Group 1: 382 epochs, 平均延迟: 0.00, 平均准确率: 70.89%
Group 2: 399 epochs, 平均延迟: 0.01, 平均准确率: 71.41%
Group 3: 384 epochs, 平均延迟: 0.

In [6]:
log_file = '/disk6T/ypguo/async-FL/时间仿真/output.log'
analysis_df = analyze_dynamic_delay_log(log_file, constant_t=5.0)

if analysis_df is not None:
    print("\n--- Analysis Results ---")
    print(analysis_df.head())
    print("\n...")
    print(analysis_df.tail())

--- Starting analysis for: /disk6T/ypguo/async-FL/时间仿真/output.log ---
Successfully read file with encoding: utf-8
--- Stage 1: Extracting data with regex ---

Regex Results:
  Found 1358 epoch blocks.
  Found 2015 selection blocks.
  Found 20150 upload blocks.

--- Stage 2: Correlating data and calculating times ---

Processing Epoch 6 (ends at 23178)
  Found 10 potential selections ending before epoch end pos 23178.
  Most recent preceding selection ends at 21265, Group ID: 4
  This is the 1-th epoch recorded for group 4.
  Using selection starting at 7045: Clients [12, 75, 25, 73, 76, 43, 67, 35, 2, 53]
  Upload search window: [7045, 21201)
  Checked 50 uploads in window.
  Found relevant uploads: Client 12 (Pos 10586): 8.22; Client 2 (Pos 10683): 8.22; Client 76 (Pos 10779): 8.22; Client 73 (Pos 11264): 8.22; Client 25 (Pos 11747): 8.22; Client 67 (Pos 11844): 8.22; Client 75 (Pos 11941): 8.22; Client 35 (Pos 12038): 8.22; Client 43 (Pos 12426): 8.22; Client 53 (Pos 12716): 8.22
  C